# LangChain Memory Classes — Updated for LangChain 1.3.6

The old tutorial used deprecated APIs (`ConversationChain`, `ConversationBufferMemory`, etc.).

**What changed:**
- `ConversationChain` → replaced by a simple `prompt | model | StrOutputParser()` chain wrapped with `RunnableWithMessageHistory`
- `ConversationBufferMemory` → replaced by `ChatMessageHistory` (stores all messages)
- `ConversationBufferWindowMemory` → replaced by `ChatMessageHistory` + `trim_messages()` (keeps last k exchanges)
- `ConversationEntityMemory` → replaced by a custom entity extraction chain (no built-in equivalent)
- `gemini-1.0-pro` → shut down, using `gemini-2.5-flash-lite`

In [1]:
!pip install -q langchain langchain-community langchain-core langchain-google-genai langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [28]:
print(model.invoke("hi").content)

[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EjQKMgEMOdbH+j/0JTyHUkgq3LZhAV9VYdOa3wSHl+koQHHRdkV2i8gmBqPpK4fWFf1ijwIs'}}]


In [29]:
print(model.invoke("hi, how are you please tell me?").content)

[{'type': 'text', 'text': "I'm doing great, thank you for asking! I'm ready to help you with whatever you need.\n\nHow are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EjQKMgEMOdbHQlW04CFNTBIBER1/Yw2y1MXI66hR8Ud5F4SglC56BElz+DnFsdbe3QWBdCxO'}}]


---
## 1. ChatMessageHistory (replaces ConversationBufferMemory)

**Old way:** `ConversationBufferMemory()` + `memory.save_context()` + `memory.load_memory_variables({})`

**New way:** `ChatMessageHistory()` + `.add_user_message()` / `.add_ai_message()` + `.messages`

In [30]:
from langchain_community.chat_message_histories import ChatMessageHistory

# Create a history object (equivalent to ConversationBufferMemory)
history = ChatMessageHistory()

# Save context (equivalent to memory.save_context)
history.add_user_message("Hi")
history.add_ai_message("What's up")

In [31]:
# View stored messages (equivalent to memory.load_memory_variables)
history.messages

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

---
## 2. Conversation Chain with Full Buffer Memory

**Old way:** `ConversationChain(llm=model, memory=ConversationBufferMemory())` + `conversation.predict(input=...)`

**New way:** Build a chain with `ChatPromptTemplate` + `model` + `StrOutputParser()`, then wrap it with `RunnableWithMessageHistory`

In [32]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# The prompt — equivalent to ConversationChain's default prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history to maintain context."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# The chain — equivalent to ConversationChain
chain = prompt | model | StrOutputParser()

# Session store — holds ChatMessageHistory per session_id
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrap the chain with automatic history management
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [33]:
# Usage — equivalent to conversation.predict(input=...)
config = {"configurable": {"session_id": "session_1"}}

conversation.invoke({"input": "Hi there!"}, config=config)

'Hello! How can I help you today?'

In [34]:
conversation.invoke(
    {"input": "Nothing much! Just tell me how to have a conversation with an AI."},
    config=config
)

'Having a conversation with an AI is a lot like talking to a very well-read, helpful assistant who has no ego and infinite patience. Here are a few tips to get the most out of our chats:\n\n### 1. Be Specific\nThe more context you provide, the better the result. Instead of saying, "Write me a story," try, "Write a short, funny story about a space-traveling cat who loves pizza."\n\n### 2. Give It a Role\nYou can "program" the AI\'s personality by telling it who to be. For example:\n*   "Act as a travel agent and help me plan a trip to Tokyo on a budget."\n*   "Talk to me like a grumpy 1940s private investigator."\n*   "Explain quantum physics like I’m a five-year-old."\n\n### 3. Iteration is Key\nDon’t worry about getting it perfect on the first try. You can treat the conversation as a back-and-forth:\n*   "That’s a bit too formal, can you make it sound friendlier?"\n*   "Can you add more details about [X]?"\n*   "I disagree with that point, let’s look at it from a different perspective

In [35]:
conversation.invoke(
    {"input": "how many tips are there, can you mention in numbers?"},
    config=config
)

"There are **6** tips in total. Here is the list numbered for you:\n\n1. **Be Specific:** Provide context to get better results.\n2. **Give It a Role:** Assign a persona or perspective to the AI.\n3. **Iteration is Key:** Treat the chat as a back-and-forth process.\n4. **Break It Down:** Divide complex tasks into smaller, manageable pieces.\n5. **Ask for Clarification:** Don't be afraid to ask the AI to re-explain or adjust its approach.\n6. **Treat it like a Partner:** Use the AI as a sounding board for brainstorming."

In [36]:
# View the full chat history (equivalent to conversation.memory.chat_memory.messages)
store["session_1"].messages

[HumanMessage(content='Hi there!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Nothing much! Just tell me how to have a conversation with an AI.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Having a conversation with an AI is a lot like talking to a very well-read, helpful assistant who has no ego and infinite patience. Here are a few tips to get the most out of our chats:\n\n### 1. Be Specific\nThe more context you provide, the better the result. Instead of saying, "Write me a story," try, "Write a short, funny story about a space-traveling cat who loves pizza."\n\n### 2. Give It a Role\nYou can "program" the AI\'s personality by telling it who to be. For example:\n*   "Act as a travel agent and help me plan a trip to Tokyo on a budget."\n*   "Talk to me like a grumpy 1940s private investigator."\n*   "E

In [37]:
conversation.invoke(
    {"input": "can you give me the 3rd tip?"},
    config=config
)

'The 3rd tip is: **Iteration is Key.**\n\nThis means you shouldn\'t worry about getting a perfect answer on the first try. You can treat the conversation as a back-and-forth process where you refine the output. For example, if I give you an answer, you can follow up by saying things like:\n\n*   "That’s a bit too formal, can you make it sound friendlier?"\n*   "Can you add more details about [specific topic]?"\n*   "I disagree with that point, let’s look at it from a different perspective."'

---
## 3. Windowed Memory (replaces ConversationBufferWindowMemory)

**Old way:** `ConversationBufferWindowMemory(k=2)` — keeps only the last k exchanges

**New way:** `ChatMessageHistory` stores everything, but we use `trim_messages()` in the chain to only pass the last k exchanges to the model

In [38]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnableLambda

# Trim to keep only the last k exchanges (k=2 means last 4 messages: 2 human + 2 AI)
k = 2

window_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history to maintain context."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

def trim_history(messages):
    """Keep only the last k exchanges (2 messages per exchange)."""
    return messages[-(k * 2):]

# Chain with trimming built in
window_chain = (
    {
        "history": lambda x: trim_history(x["history"]),
        "input": lambda x: x["input"],
    }
    | window_prompt
    | model
    | StrOutputParser()
)

window_store = {}

def get_window_history(session_id: str):
    if session_id not in window_store:
        window_store[session_id] = ChatMessageHistory()
    return window_store[session_id]

conversation_window = RunnableWithMessageHistory(
    window_chain,
    get_window_history,
    input_messages_key="input",
    history_messages_key="history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [39]:
window_config = {"configurable": {"session_id": "window_1"}}

conversation_window.invoke({"input": "Hi, what's up?"}, config=window_config)

'Not much! Just here and ready to help. How are things going with you? Is there anything I can assist you with today?'

In [40]:
conversation_window.invoke(
    {"input": "how we can talk with AI? give me 5 points"},
    config=window_config
)

'Talking to an AI is a bit like learning to communicate with a very literal-minded librarian who has read everything but has no personal intuition. \n\nHere are 5 key points on how to communicate effectively with an AI:\n\n### 1. Be Specific and Clear\nAI works best when you provide precise instructions. Instead of asking "Write something about history," try "Write a 300-word summary of the causes of the French Revolution for a high school student." The more context you provide, the better the result will be.\n\n### 2. Provide Context\nAI doesn\'t know your background or your current situation unless you tell it. If you need help with a project, explain the goal, the intended audience, and the tone you are aiming for (e.g., "Act as a professional editor and fix the grammar in this email while keeping the tone friendly").\n\n### 3. Use an Iterative Process (Refine)\nDon’t expect perfection on the first try. If the AI gives you an answer that isn\'t quite right, tell it what to change. Y

In [41]:
conversation_window.invoke(
    {"input": "what allows AI to 'see' and 'interpret' images?"},
    config=window_config
)

'To understand how AI "sees" images, it helps to realize that **AI doesn\'t actually "see" in the way humans do.** It doesn\'t have eyes, and it doesn\'t recognize a "cat" because it understands what a cat is biologically.\n\nInstead, AI interprets images through a combination of **mathematical data** and **pattern recognition**. Here is the breakdown of how that works:\n\n### 1. Turning Images into Numbers (Pixels)\nAt the most basic level, a computer sees an image as a giant grid of numbers. Every digital image is made up of millions of tiny squares called pixels. Each pixel is assigned a numerical value representing its color and brightness. For example, a black-and-white image is just a grid of numbers between 0 (black) and 255 (white). The AI "looks" at these grids to identify patterns.\n\n### 2. Convolutional Neural Networks (CNNs)\nThe primary technology behind computer vision is something called a **Convolutional Neural Network**. Think of this like a series of filters that sca

In [42]:
# This should fail to recall — the 5 tips message has been trimmed out of the window
conversation_window.invoke(
    {"input": "can you tell me how many tips you generated in the previous to previous message?"},
    config=window_config
)

'In the message where I provided tips on how to talk to an AI, I generated **5 tips**.'

In [43]:
# Full history is still stored, but only last k=2 exchanges are sent to the model
print(f"Total messages stored: {len(window_store['window_1'].messages)}")
print(f"Messages sent to model: {k * 2}")
window_store["window_1"].messages

Total messages stored: 8
Messages sent to model: 4


[HumanMessage(content="Hi, what's up?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Not much! Just here and ready to help. How are things going with you? Is there anything I can assist you with today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='how we can talk with AI? give me 5 points', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Talking to an AI is a bit like learning to communicate with a very literal-minded librarian who has read everything but has no personal intuition. \n\nHere are 5 key points on how to communicate effectively with an AI:\n\n### 1. Be Specific and Clear\nAI works best when you provide precise instructions. Instead of asking "Write something about history," try "Write a 300-word summary of the causes of the French Revolution for a high school student." The more context you provide, the better the result will be.\n\n### 2. Provide Context\nAI doesn\'t know you

---
## 4. Entity Memory (replaces ConversationEntityMemory)

**Old way:** `ConversationEntityMemory(llm=model)` — automatically extracted entities from conversation

**New way:** No built-in equivalent in LangChain 1.x. We build a custom entity extraction step that calls the LLM to extract/update entities after each exchange, then injects them into the system prompt.

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Entity store — simple dict like the old entity_store.store
entity_store = {}

# Chain to extract entities from a conversation turn
entity_extraction_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an entity extraction system. Given the conversation below, "
     "extract or update information about people, organizations, and projects mentioned. "
     "Return ONLY a Python dict literal (no markdown, no backticks) mapping entity names to descriptions. "
     "If no entities found, return an empty dict: {{}}"
     "\n\nExisting entities: {entities}"),
    ("human", "User said: {user_input}\nAI said: {ai_output}\n\nExtract/update entities:"),
])

entity_chain = entity_extraction_prompt | model | StrOutputParser()


def update_entities(user_input: str, ai_output: str):
    """Extract entities from the latest exchange and update the store."""
    result = entity_chain.invoke({
        "entities": str(entity_store),
        "user_input": user_input,
        "ai_output": ai_output,
    })
    try:
        new_entities = eval(result)
        if isinstance(new_entities, dict):
            entity_store.update(new_entities)
    except:
        pass  # if parsing fails, skip the update

In [45]:
# Conversation chain that uses entity context
entity_conv_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. You know the following about entities mentioned in this conversation:\n"
     "{entity_context}\n\n"
     "Use this information to provide informed responses."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

entity_conv_chain = entity_conv_prompt | model | StrOutputParser()

entity_history = ChatMessageHistory()


def entity_chat(user_input: str) -> str:
    """Chat with entity memory — equivalent to old conversation.predict(input=...)"""
    # Build entity context string
    entity_context = "\n".join(
        f"- {name}: {desc}" for name, desc in entity_store.items()
    ) if entity_store else "No entities known yet."

    # Get response
    response = entity_conv_chain.invoke({
        "entity_context": entity_context,
        "history": entity_history.messages,
        "input": user_input,
    })

    # Save to history
    entity_history.add_user_message(user_input)
    entity_history.add_ai_message(response)

    # Extract and update entities
    update_entities(user_input, response)

    return response

In [46]:
entity_chat("Deven & Sam are working on a hackathon project")

"That sounds like an exciting project! Hackathons are a great way to build something cool in a short amount of time.\n\nDo you know what they are building, or what kind of project they're focused on? I’d love to hear more about it—maybe I can help with brainstorming, troubleshooting code, or refining their pitch!"

In [47]:
# View extracted entities (equivalent to conversation.memory.entity_store.store)
from pprint import pprint
pprint(entity_store)

{'Deven': 'Working on a hackathon project',
 'Sam': 'Working on a hackathon project'}


In [48]:
entity_chat("They are trying to add more complex memory structures to Langchain")

'That is an ambitious and highly relevant challenge! Integrating more complex memory structures into LangChain can significantly improve an agent\'s ability to handle long-term context, summarization, or even vector-based retrieval beyond the standard `ConversationBufferMemory`.\n\nAre they looking to implement a specific architecture, or are they exploring a few different approaches? Depending on their goal, here are a few areas where they might want to focus:\n\n1.  **Graph-based Memory:** Instead of just a list of messages, are they trying to build a Knowledge Graph memory (using something like Neo4j or NetworkX) so the LLM can query relationships between entities?\n2.  **Hierarchical/Multi-Level Memory:** Are they working on a "short-term vs. long-term" architecture where the agent summarizes recent interactions into a persistent summary buffer while keeping the raw data in a vector store?\n3.  **Entity-Centric Memory:** Are they focusing on tracking the state of specific entities 

In [49]:
entity_chat("They are adding in a key-value store for entities mentioned so far in the conversation.")

'That sounds like a great use case! Implementing an entity-based key-value store essentially allows them to build a **dynamic "state" or "fact base"** that the model can reference, which is far more efficient than searching through a massive chat history buffer.\n\nSince they are using LangChain for this, here are a few things they might want to consider for their implementation:\n\n### 1. The Extraction Pipeline\nTo make this work, they’ll need a robust "extractor." They could set up an extraction chain that runs in parallel or as a post-process step after each user turn.\n*   **The Prompt:** Use an LLM to extract `(Entity, Attribute, Value)` triplets.\n*   **Structured Output:** They should definitely use LangChain’s **`with_structured_output`** (using Pydantic models) to ensure the extractor always returns a clean dictionary or object that can be easily dumped into their KV store.\n\n### 2. Choosing the KV Store\nSince it’s a hackathon, they have a few options for the backend:\n*   

In [50]:
entity_chat("What do you know about Deven & Sam?")

'I know that Deven and Sam are currently working together on a hackathon project. Their project focuses on implementing complex memory structures within the LangChain framework. Specifically, they are building an **entity-based key-value store** to handle dynamic state management during conversations. \n\nThey are leveraging LangChain’s `BaseMemory` classes and utilizing its structured output capabilities (likely Pydantic models) to perform the entity extraction necessary to populate that key-value store. \n\nIt sounds like they are tackling a sophisticated piece of architecture—are they looking for help with the `BaseMemory` integration or perhaps the logic for how the agent merges new entity updates into the store?'

In [51]:
pprint(entity_store)

{'Deven': 'Working on a hackathon project involving complex memory structures '
          'for LangChain, currently implementing an entity-based key-value '
          'store for dynamic state management using BaseMemory and structured '
          'output.',
 'LangChain': 'Framework being used for the hackathon project to implement '
              'complex memory structures, specifically utilizing structured '
              'output and BaseMemory classes for entity extraction.',
 'Sam': 'Working on a hackathon project involving complex memory structures '
        'for LangChain, currently implementing an entity-based key-value store '
        'for dynamic state management using BaseMemory and structured output.'}


In [52]:
entity_chat("Sam is the founder of a company called Daimon.")

'That\'s interesting context! Knowing that Sam is the founder of **Daimon** adds a layer of practical importance to this hackathon project. If Daimon is an AI-driven venture or a company focused on agentic workflows, the work Sam and Deven are doing on entity-based memory isn\'t just a prototype—it’s likely a core component for a real-world product.\n\nEntity-based state management is a "holy grail" for AI agents, as it moves them away from being stateless chatbots toward being true assistants that can maintain a consistent "world model."\n\nSince they are dealing with **`BaseMemory`** and **structured output** for this, they are effectively building a custom "state layer." Depending on what Daimon does, they might eventually need to consider:\n\n*   **Conflict Resolution:** If the agent discovers contradictory information about an entity, how does the memory store handle it? (e.g., does it use a timestamp, or does the LLM decide which fact is more recent/reliable?)\n*   **Context Wind

In [53]:
pprint(entity_store)

{'Daimon': 'A company founded by Sam, likely focused on AI-driven ventures or '
           'agentic workflows.',
 'Deven': 'Working on a hackathon project involving complex memory structures '
          'for LangChain, currently implementing an entity-based key-value '
          'store for dynamic state management using BaseMemory and structured '
          'output.',
 'LangChain': 'Framework being used for the hackathon project to implement '
              'complex memory structures, specifically utilizing structured '
              'output and BaseMemory classes for entity extraction.',
 'Sam': 'Founder of Daimon; working on a hackathon project involving complex '
        'memory structures for LangChain, currently implementing an '
        'entity-based key-value store for dynamic state management using '
        'BaseMemory and structured output.'}


In [54]:
entity_chat("What do you know about Sam?")

"Based on our conversation, here is what I know about **Sam**:\n\n*   **Role:** Sam is the founder of a company called **Daimon**, which is likely focused on AI-driven ventures or agentic workflows.\n*   **Current Activity:** Sam is participating in a hackathon alongside Deven.\n*   **Technical Project:** They are working on implementing advanced, complex memory structures within the **LangChain** framework.\n*   **Technical Focus:** Specifically, Sam is helping build an **entity-based key-value store** designed for dynamic state management. To achieve this, they are using LangChain's `BaseMemory` class and structured output (leveraging Pydantic models) to perform entity extraction throughout conversations. \n\nIt sounds like Sam is taking a very practical, engineering-heavy approach to solving one of the biggest hurdles in AI agent development: persistent and intelligent memory."